# DocSensei: Enterprise Multi-Document Adaptive RAG Assistant

**DocSensei** is a production-grade Retrieval-Augmented Generation assistant that:

- Ingests multiple **PDF / DOCX / PPTX / TXT** files in a single session
- Answers questions **only** from the uploaded documents, with **page + file citations**
- Uses an **Adaptive RAG Decision Layer** to route queries (simple / complex / ambiguous) to the right retrieval strategy
- Combines **dense (Pinecone) + sparse (BM25) hybrid retrieval**
- Applies **contextual compression**, **query rewriting**, and **conversation memory**
- Serves everything through a modern **Gradio** chat interface

**Tech stack:** Python, LangChain, Pinecone, HuggingFace (`all-MiniLM-L6-v2`), Groq `llama-3.3-70b-versatile`, Gradio, `python-dotenv`.

> Run cells top to bottom in Google Colab. You will need a `.env` file (or Colab secrets) with `PINECONE_API_KEY` and `GROQ_API_KEY`.


## 2. Install Dependencies

In [1]:
# %%capture
# !pip install -q groq pinecone-client rank_bm25 sentence-transformers
# !pip install -q pypdf python-docx python-pptx
# !pip install -q pytesseract pdf2image pillow
# !pip install -q python-dotenv
# !apt-get -qq install -y poppler-utils tesseract-ocr


## 3. Imports

In [2]:
import io
import os
import re
import json
import time
import logging
import unicodedata
import uuid
from collections import deque
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple, Any

from dotenv import load_dotenv

# ---------------------------------------------------------------------------
# Third-party libraries -- identical stack to the deployed app.py. Dense
# retrieval, chunking, and document loading are all dependency-light, raw
# implementations (no LangChain) so the notebook and app.py share the exact
# same code paths.
# ---------------------------------------------------------------------------
from pypdf import PdfReader
from docx import Document as DocxDocument
from docx.oxml.ns import qn
from pptx import Presentation
from pptx.shapes.group import GroupShape

from sentence_transformers import CrossEncoder, SentenceTransformer
from rank_bm25 import BM25Okapi
from groq import Groq
from pinecone import Pinecone, ServerlessSpec

# Optional OCR fallback for scanned / empty-text PDF pages. Guarded with a
# try/except so the notebook still runs end-to-end even if these packages
# or the underlying system binaries (poppler, tesseract) are not installed.
try:
    import pytesseract
    from pdf2image import convert_from_path
    OCR_LIBS_AVAILABLE = True
except ImportError:
    OCR_LIBS_AVAILABLE = False

#import gradio as gr


C:\Users\monis\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 4. Configuration

In [3]:
class Config:
    # Embeddings
    EMBEDDING_MODEL: str = "sentence-transformers/all-MiniLM-L6-v2"
    EMBEDDING_DIM: int = 384

    # Reranking
    RERANKER_MODEL: str = "cross-encoder/ms-marco-MiniLM-L-6-v2"
    RERANK_CANDIDATE_POOL: int = 15   # how many hybrid results to feed the reranker
    RERANK_TOP_N: int = 8             # how many reranked chunks survive to compression
    COMPRESSION_TOP_N: int = 5        # only compress the top-5 reranked chunks

    # Groq
    GROQ_MODEL: str = "llama-3.3-70b-versatile"
    GROQ_TEMPERATURE: float = 0.1

    # Pinecone
    PINECONE_INDEX_NAME: str = "docsensei-index"
    PINECONE_CLOUD: str = "aws"
    PINECONE_REGION: str = "us-east-1"

    # Chunking
    CHUNK_SIZE: int = 1000
    CHUNK_OVERLAP: int = 150

    # Hybrid retrieval fusion weights
    DENSE_WEIGHT: float = 0.6
    BM25_WEIGHT: float = 0.4

    MEMORY_WINDOW: int = 6  # number of past conversation turns kept verbatim

    ENABLE_OCR_FALLBACK: bool = True
    OCR_DPI: int = 200
    OCR_MIN_CHARS_PER_PAGE: int = 20  # below this, a PDF page is treated as "empty" -> OCR


config = Config()

# Adaptive router labels and their retrieval depth -- identical to app.py.
QUERY_LABELS = (
    "Simple",
    "Complex",
    "Comparison",
    "Summarization",
    "Reasoning",
    "Multi-document",
    "Follow-up",
)

ROUTER_TOP_K = {
    "Simple": 3,
    "Complex": 8,
    "Comparison": 8,
    "Summarization": 10,
    "Reasoning": 8,
    "Multi-document": 10,
    "Follow-up": 5,
}


## 5. API Setup (.env)


```


In [4]:
load_dotenv()

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if not PINECONE_API_KEY or not GROQ_API_KEY:
    raise EnvironmentError(
        "Missing PINECONE_API_KEY or GROQ_API_KEY. "
        "Create a .env file with both keys before proceeding."
    )


## 6. Logging

In [5]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
)
logger = logging.getLogger("DocSensei")
logger.info("Logging initialized.")


2026-07-10 08:46:02,589 | INFO     | DocSensei | Logging initialized.


## 7. Utility Functions

In [6]:
def clean_text(text: str) -> str:
    """Normalize unicode, collapse whitespace, and remove duplicate blank
    lines and empty paragraphs, while preserving useful structural markers
    (headings, bullets) produced by the loaders below. Identical logic to
    app.py's clean_text.
    """
    if not text:
        return ""
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    lines = [line.strip() for line in text.split("\n")]
    deduped_lines = []
    previous_blank = False
    for line in lines:
        if line == "":
            if previous_blank:
                continue
            previous_blank = True
        else:
            previous_blank = False
        deduped_lines.append(line)
    return "\n".join(deduped_lines).strip()


def file_extension(filename: str) -> str:
    return os.path.splitext(filename)[1].lower()


def timeit(label: str):
    """Simple context manager for timing and logging pipeline stages."""
    class _Timer:
        def __enter__(self):
            self.start = time.time()
            return self

        def __exit__(self, *exc):
            logger.info(f"{label} took {time.time() - self.start:.2f}s")

    return _Timer()


## 8. Document Loaders

In [7]:
SUPPORTED_EXTENSIONS = {".pdf", ".docx", ".pptx", ".txt"}


def ocr_is_available() -> bool:
    """Check whether the OCR system binaries (tesseract + poppler) are
    actually usable, not just whether the Python packages import cleanly.
    """
    if not OCR_LIBS_AVAILABLE or not config.ENABLE_OCR_FALLBACK:
        return False
    try:
        pytesseract.get_tesseract_version()
        return True
    except Exception as e:
        logger.warning(
            f"OCR fallback disabled (tesseract/poppler not found): {e}. "
            "Scanned/image-only PDF pages will yield empty text."
        )
        return False


OCR_AVAILABLE = ocr_is_available()


def _ocr_pdf_page(filepath: str, page_number: int) -> str:
    """Render a single PDF page to an image and OCR it. Returns "" on failure."""
    if not OCR_AVAILABLE:
        return ""
    try:
        images = convert_from_path(
            filepath, dpi=config.OCR_DPI, first_page=page_number, last_page=page_number
        )
        if not images:
            return ""
        return pytesseract.image_to_string(images[0]) or ""
    except Exception as e:
        logger.warning(f"OCR fallback failed for {filepath} page {page_number}: {e}")
        return ""


def load_pdf(filepath: str) -> List[Dict[str, Any]]:
    """Extract text from a PDF file, page by page, using pypdf. Pages with
    little or no extractable text (e.g. scanned images) are transparently
    retried through OCR when available.
    """
    pages: List[Dict[str, Any]] = []
    ocr_used_count = 0
    try:
        reader = PdfReader(filepath)
        for page_number, page in enumerate(reader.pages, start=1):
            try:
                text = page.extract_text() or ""
            except Exception:
                logger.exception(f"Native text extraction failed on page {page_number} of {filepath}")
                text = ""

            if len(text.strip()) < config.OCR_MIN_CHARS_PER_PAGE and OCR_AVAILABLE:
                ocr_text = _ocr_pdf_page(filepath, page_number)
                if len(ocr_text.strip()) > len(text.strip()):
                    text = ocr_text
                    ocr_used_count += 1

            if text.strip():
                pages.append({"text": text, "page": page_number})

        logger.info(
            f"Loaded PDF '{os.path.basename(filepath)}' with {len(pages)} pages "
            f"(OCR used on {ocr_used_count} page(s))"
        )
    except Exception:
        logger.exception(f"Failed to load PDF: {filepath}")
    return pages


def _iter_docx_headers_footers(doc):
    """Yield (label, text) pairs for every header/footer across all sections."""
    for section_index, section in enumerate(doc.sections, start=1):
        for label, part in (
            (f"header (section {section_index})", section.header),
            (f"first-page header (section {section_index})", section.first_page_header),
            (f"footer (section {section_index})", section.footer),
            (f"first-page footer (section {section_index})", section.first_page_footer),
        ):
            try:
                text = "\n".join(p.text for p in part.paragraphs if p.text.strip())
            except Exception:
                text = ""
            if text.strip():
                yield label, text


def _extract_docx_textboxes(doc) -> List[str]:
    """Extract text from DrawingML/VML text boxes via raw XML traversal.
    python-docx has no first-class API for text boxes, so we walk the
    underlying XML looking for <w:txbxContent> elements.
    """
    textbox_texts: List[str] = []
    try:
        body = doc.element.body
        for txbx in body.iter(qn("w:txbxContent")):
            paragraphs = txbx.iter(qn("w:p"))
            texts = []
            for p in paragraphs:
                runs = p.iter(qn("w:t"))
                run_text = "".join(t.text or "" for t in runs)
                if run_text.strip():
                    texts.append(run_text)
            joined = "\n".join(texts)
            if joined.strip():
                textbox_texts.append(joined)
    except Exception:
        logger.exception("Text box extraction failed for DOCX")
    return textbox_texts


def load_docx(filepath: str) -> List[Dict[str, Any]]:
    """Extract text from a DOCX file: paragraphs, headings, tables, headers,
    footers, and (best-effort) text boxes. Empty objects are skipped.
    """
    pages: List[Dict[str, Any]] = []
    try:
        document = DocxDocument(filepath)
        parts: List[str] = []

        for para in document.paragraphs:
            text = para.text.strip()
            if not text:
                continue
            style_name = (para.style.name or "").lower() if para.style else ""
            if "heading" in style_name or "title" in style_name:
                parts.append(f"## {text}")
            else:
                parts.append(text)

        for table_index, table in enumerate(document.tables, start=1):
            table_rows = []
            for row in table.rows:
                cells = [cell.text.strip() for cell in row.cells if cell.text.strip()]
                if cells:
                    table_rows.append(" | ".join(cells))
            if table_rows:
                parts.append(f"[Table {table_index}]\n" + "\n".join(table_rows))

        for label, text in _iter_docx_headers_footers(document):
            parts.append(f"[{label}] {text}")

        for i, textbox_text in enumerate(_extract_docx_textboxes(document), start=1):
            parts.append(f"[Text box {i}] {textbox_text}")

        full_text = "\n".join(p for p in parts if p.strip())
        if full_text.strip():
            pages.append({"text": full_text, "page": 1})
        logger.info(f"Loaded DOCX '{os.path.basename(filepath)}' ({len(parts)} content blocks)")
    except Exception:
        logger.exception(f"Failed to load DOCX: {filepath}")
    return pages


def _iter_pptx_shapes(shapes):
    """Recursively yield shapes, descending into grouped shapes."""
    for shape in shapes:
        if isinstance(shape, GroupShape) or getattr(shape, "shape_type", None) == 6:
            try:
                yield from _iter_pptx_shapes(shape.shapes)
                continue
            except Exception:
                pass
        yield shape


def _extract_pptx_chart_labels(shape) -> List[str]:
    """Extract category/series labels from a chart shape, if present."""
    labels: List[str] = []
    try:
        if not shape.has_chart:
            return labels
        chart = shape.chart
        for plot in chart.plots:
            try:
                categories = [str(c) for c in plot.categories if c is not None]
                labels.extend(categories)
            except Exception:
                pass
            for series in plot.series:
                name = getattr(series, "name", None)
                if name:
                    labels.append(str(name))
    except Exception:
        logger.exception("Chart label extraction failed")
    return labels


def load_pptx(filepath: str) -> List[Dict[str, Any]]:
    """Extract text from a PPTX file per slide: titles, placeholders, bullet
    lists, grouped shapes, text boxes, speaker notes, and chart labels.
    """
    pages: List[Dict[str, Any]] = []
    try:
        presentation = Presentation(filepath)
        for slide_number, slide in enumerate(presentation.slides, start=1):
            slide_parts: List[str] = []

            for shape in _iter_pptx_shapes(slide.shapes):
                if getattr(shape, "has_text_frame", False):
                    is_title = False
                    try:
                        is_title = shape.placeholder_format is not None and shape.placeholder_format.idx == 0
                    except Exception:
                        is_title = False
                    for paragraph in shape.text_frame.paragraphs:
                        line = "".join(run.text for run in paragraph.runs).strip()
                        if not line and paragraph.text.strip():
                            line = paragraph.text.strip()
                        if line:
                            slide_parts.append(f"# {line}" if is_title else line)

                chart_labels = _extract_pptx_chart_labels(shape)
                if chart_labels:
                    slide_parts.append("[Chart] " + ", ".join(chart_labels))

            try:
                if slide.has_notes_slide:
                    notes_text = slide.notes_slide.notes_text_frame.text.strip()
                    if notes_text:
                        slide_parts.append(f"[Speaker notes] {notes_text}")
            except Exception:
                logger.exception(f"Speaker notes extraction failed on slide {slide_number}")

            slide_text = "\n".join(slide_parts)
            if slide_text.strip():
                pages.append({"text": slide_text, "page": slide_number})

        logger.info(f"Loaded PPTX '{os.path.basename(filepath)}' with {len(pages)} slides of text")
    except Exception:
        logger.exception(f"Failed to load PPTX: {filepath}")
    return pages


def load_txt(filepath: str) -> List[Dict[str, Any]]:
    """Load a plain text file."""
    pages: List[Dict[str, Any]] = []
    try:
        with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
            text = f.read()
        if text.strip():
            pages.append({"text": text, "page": 1})
        logger.info(f"Loaded TXT '{os.path.basename(filepath)}'")
    except Exception:
        logger.exception(f"Failed to load TXT: {filepath}")
    return pages


LOADER_MAP = {
    ".pdf": load_pdf,
    ".docx": load_docx,
    ".pptx": load_pptx,
    ".txt": load_txt,
}


def load_document(filepath: str) -> Tuple[str, str, List[Dict[str, Any]]]:
    """Load a single document into (filename, file_type, pages), where each
    page dict is {"text": ..., "page": ...}. Raises ValueError for
    unsupported extensions.
    """
    ext = file_extension(filepath)
    filename = os.path.basename(filepath)

    if ext not in SUPPORTED_EXTENSIONS:
        raise ValueError(f"Unsupported file type: {ext}")

    logger.info(f"Loading {filename} ({ext})")
    loader = LOADER_MAP[ext]
    pages = loader(filepath)
    return filename, ext.lstrip("."), pages


def load_documents(filepaths: List[str]) -> Dict[str, Tuple[str, List[Dict[str, Any]]]]:
    """Load multiple files. If one file fails to load, log it and continue
    processing the remaining files instead of aborting the whole batch.
    Returns {filename: (file_type, pages)}.
    """
    loaded: Dict[str, Tuple[str, List[Dict[str, Any]]]] = {}
    for fp in filepaths:
        try:
            filename, file_type, pages = load_document(fp)
            if not pages:
                logger.warning(f"No extractable text in file: {filename}")
                continue
            loaded[filename] = (file_type, pages)
        except Exception:
            logger.exception(f"Failed to load '{fp}', skipping this file.")
    logger.info(f"Loaded {len(loaded)} file(s) out of {len(filepaths)} requested.")
    return loaded


## 9. Metadata Extraction

In [8]:
def build_raw_documents(loaded: Dict[str, Tuple[str, List[Dict[str, Any]]]]) -> Dict[str, str]:
    """Concatenate the cleaned page/slide text of each file into one full
    document string, exactly as app.py does before chunking:
        full_text = "\n\n".join(clean_text(p["text"]) for p in pages)
    These full texts are later used for structured summarization and
    suggested-question generation.
    """
    raw_documents: Dict[str, str] = {}
    for filename, (file_type, pages) in loaded.items():
        full_text = "\n\n".join(clean_text(p["text"]) for p in pages)
        if full_text.strip():
            raw_documents[filename] = full_text
        else:
            logger.warning(f"No extractable text in file after cleaning: {filename}")
    return raw_documents


## 10. Chunking

In [9]:
BULLET_PREFIXES = ("- ", "* ", "\u2022 ", "\u25cf ")


def _is_heading_line(line: str) -> bool:
    return line.startswith("#") or line.startswith("[Table") or line.startswith("[Speaker notes")


def _is_bullet_line(line: str) -> bool:
    return line.lstrip().startswith(BULLET_PREFIXES)


def _presegment_structure_aware(text: str) -> List[str]:
    """Group lines into structural blocks so headings stay attached to the
    paragraph that follows them, and consecutive bullet-list lines stay
    together as one block, before the recursive splitter ever sees them.
    """
    lines = text.split("\n")
    blocks: List[str] = []
    current: List[str] = []
    current_is_bullets = False

    def flush():
        if current:
            blocks.append("\n".join(current))
            current.clear()

    for line in lines:
        if not line.strip():
            flush()
            current_is_bullets = False
            continue
        if _is_heading_line(line):
            flush()
            current.append(line)
            current_is_bullets = False
            continue
        if _is_bullet_line(line):
            if current and not current_is_bullets and not _is_heading_line(current[-1]):
                flush()
            current.append(line)
            current_is_bullets = True
            continue
        if current_is_bullets:
            flush()
        current.append(line)
        current_is_bullets = False

    flush()
    return [b for b in blocks if b.strip()]


def _split_by_separator(text: str, separator: str) -> List[str]:
    if separator == "":
        return list(text)
    return text.split(separator)


def recursive_character_split(
    text: str, chunk_size: int = None, chunk_overlap: int = None
) -> List[str]:
    """A dependency-free recursive character splitter modeled after
    LangChain's RecursiveCharacterTextSplitter, using a cascading list of
    separators from coarse to fine granularity. Structural blocks
    (headings + following paragraph, bullet-list runs) are pre-merged so
    the splitter is much less likely to sever a heading from its body text
    or break a bullet list mid-list. Identical algorithm to app.py.
    """
    chunk_size = chunk_size or config.CHUNK_SIZE
    chunk_overlap = chunk_overlap if chunk_overlap is not None else config.CHUNK_OVERLAP
    separators = ["\n\n", "\n", ". ", " ", ""]

    def _split(text_block: str, seps: List[str]) -> List[str]:
        if len(text_block) <= chunk_size:
            return [text_block] if text_block.strip() else []

        if not seps:
            return [text_block[i:i + chunk_size] for i in range(0, len(text_block), chunk_size)]

        sep = seps[0]
        pieces = _split_by_separator(text_block, sep)

        chunks: List[str] = []
        current = ""
        for piece in pieces:
            candidate = current + (sep if current else "") + piece if sep else current + piece
            if len(candidate) <= chunk_size:
                current = candidate
            else:
                if current.strip():
                    chunks.append(current)
                if len(piece) > chunk_size:
                    chunks.extend(_split(piece, seps[1:]))
                    current = ""
                else:
                    current = piece
        if current.strip():
            chunks.append(current)
        return chunks

    blocks = _presegment_structure_aware(text)
    merged_blocks: List[str] = []
    current = ""
    for block in blocks:
        candidate = (current + "\n\n" + block) if current else block
        if len(candidate) <= chunk_size:
            current = candidate
        else:
            if current:
                merged_blocks.append(current)
            current = block
    if current:
        merged_blocks.append(current)

    raw_chunks: List[str] = []
    for block in merged_blocks:
        raw_chunks.extend(_split(block, separators))

    overlapped_chunks = []
    for i, chunk in enumerate(raw_chunks):
        if i == 0 or chunk_overlap <= 0:
            overlapped_chunks.append(chunk)
        else:
            prev_tail = raw_chunks[i - 1][-chunk_overlap:]
            overlapped_chunks.append((prev_tail + chunk)[:chunk_size + chunk_overlap])
    return [c.strip() for c in overlapped_chunks if c.strip()]


def build_chunks_for_file(filename: str, file_type: str, pages: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """Convert loaded page/slide text blocks into metadata-rich chunks,
    identical to app.py's build_chunks_for_file (cleans each page, splits,
    and preserves source_file / page / file_type metadata per chunk).
    """
    chunks = []
    for page_entry in pages:
        cleaned = clean_text(page_entry["text"])
        if not cleaned:
            continue
        text_pieces = recursive_character_split(cleaned)
        for piece in text_pieces:
            chunk_id = str(uuid.uuid4())
            chunks.append({
                "chunk_id": chunk_id,
                "text": piece,
                "source_file": filename,
                "page": page_entry["page"],
                "file_type": file_type,
            })
    logger.info(f"Built {len(chunks)} chunks for file '{filename}'")
    return chunks


def chunk_documents(loaded: Dict[str, Tuple[str, List[Dict[str, Any]]]]) -> List[Dict[str, Any]]:
    """Build chunks for every loaded file and concatenate into one list."""
    all_chunks: List[Dict[str, Any]] = []
    for filename, (file_type, pages) in loaded.items():
        all_chunks.extend(build_chunks_for_file(filename, file_type, pages))
    logger.info(f"Created {len(all_chunks)} chunks total from {len(loaded)} file(s).")
    return all_chunks


## 11. Embeddings

In [10]:
embedding_model = SentenceTransformer(config.EMBEDDING_MODEL)
logger.info(f"Loaded embedding model: {config.EMBEDDING_MODEL}")


def embed_texts(texts: List[str]) -> List[List[float]]:
    """Embed a list of texts, batched and L2-normalized so cosine similarity
    behaves as expected in Pinecone. Identical to app.py's embed_texts.
    """
    if not texts:
        return []
    embeddings = embedding_model.encode(
        texts,
        batch_size=32,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    return embeddings.tolist()


2026-07-10 08:46:02,702 | INFO     | sentence_transformers.base.model | No device provided, using cpu
2026-07-10 08:46:04,262 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-07-10 08:46:04,265 | WARNING  | huggingface_hub.utils._http | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-07-10 08:46:04,293 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json "HTTP/1.1 200 OK"
2026-07-10 08:46:04,727 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-07-10 08:46:04,751 | INFO     | httpx | HTTP Request: HEAD https://huggi

## 12. Pinecone Vector Database

In [11]:
pc = Pinecone(api_key=PINECONE_API_KEY)

existing_indexes = [i["name"] for i in pc.list_indexes()]
if config.PINECONE_INDEX_NAME not in existing_indexes:
    pc.create_index(
        name=config.PINECONE_INDEX_NAME,
        dimension=config.EMBEDDING_DIM,
        metric="cosine",
        spec=ServerlessSpec(cloud=config.PINECONE_CLOUD, region=config.PINECONE_REGION),
    )
    while not pc.describe_index(config.PINECONE_INDEX_NAME).status["ready"]:
        time.sleep(1)
    logger.info(f"Created Pinecone index '{config.PINECONE_INDEX_NAME}'.")
else:
    logger.info(f"Using existing Pinecone index '{config.PINECONE_INDEX_NAME}'.")

pinecone_index = pc.Index(config.PINECONE_INDEX_NAME)


def delete_all_vectors() -> None:
    """Wipe every previously indexed vector before a new upload. Every
    upload must start a completely new RAG session: nothing from a
    previous document set should remain retrievable afterward.
    """
    try:
        pinecone_index.delete(delete_all=True)
        logger.info("Pinecone: deleted all previous vectors.")
    except Exception as e:
        # A brand-new or already-empty serverless index can raise here on
        # some Pinecone SDK versions -- this is safe to ignore.
        logger.warning(f"Pinecone delete_all warning (likely an empty index): {e}")


def upsert_chunks_to_pinecone(chunks: List[Dict[str, Any]]) -> None:
    """Embed and upsert chunks into Pinecone in batches. Identical logic to
    app.py's upsert_chunks_to_pinecone.
    """
    if not chunks:
        logger.warning("No chunks to index.")
        return

    texts = [c["text"] for c in chunks]
    vectors_values = embed_texts(texts)

    vectors = []
    for chunk, values in zip(chunks, vectors_values):
        vectors.append({
            "id": chunk["chunk_id"],
            "values": values,
            "metadata": {
                "text": chunk["text"][:4000],  # metadata size guard
                "source_file": chunk["source_file"],
                "page": chunk["page"],
                "file_type": chunk["file_type"],
                "chunk_id": chunk["chunk_id"],
            },
        })

    batch_size = 100
    for i in range(0, len(vectors), batch_size):
        batch = vectors[i:i + batch_size]
        pinecone_index.upsert(vectors=batch)
    logger.info(f"Indexed {len(vectors)} chunks into Pinecone.")


def dense_retrieve(query: str, top_k: int = 5) -> List[Dict[str, Any]]:
    """Retrieve the top_k most relevant chunks via dense vector search.
    Identical logic to app.py's dense_retrieve.
    """
    query_vector = embed_texts([query])[0]
    result = pinecone_index.query(vector=query_vector, top_k=top_k, include_metadata=True)
    matches = result.get("matches", []) if isinstance(result, dict) else result.matches
    retrieved = []
    for match in matches:
        metadata = match["metadata"] if isinstance(match, dict) else match.metadata
        score = match["score"] if isinstance(match, dict) else match.score
        retrieved.append({
            "chunk_id": metadata.get("chunk_id"),
            "text": metadata.get("text", ""),
            "source_file": metadata.get("source_file"),
            "page": metadata.get("page"),
            "file_type": metadata.get("file_type"),
            "score": float(score),
            "retriever": "dense",
        })
    return retrieved


2026-07-10 08:46:18,279 | INFO     | DocSensei | Using existing Pinecone index 'docsensei-index'.


## 13. BM25 Retriever

In [12]:
def _tokenize(text: str) -> List[str]:
    return re.findall(r"[a-z0-9]+", text.lower())


bm25_index: Optional[BM25Okapi] = None
bm25_corpus_tokens: List[List[str]] = []


def build_bm25_index(chunks: List[Dict[str, Any]]) -> Optional[BM25Okapi]:
    """Build (or rebuild) the in-memory BM25 index from the current chunks.
    Called automatically after every upload so BM25 always reflects the
    latest document set. Identical logic to app.py's build_bm25_index.
    """
    global bm25_index, bm25_corpus_tokens
    if not chunks:
        bm25_index = None
        bm25_corpus_tokens = []
        return None
    bm25_corpus_tokens = [_tokenize(chunk["text"]) for chunk in chunks]
    bm25_index = BM25Okapi(bm25_corpus_tokens)
    logger.info(f"Built BM25 index over {len(chunks)} chunks.")
    return bm25_index


def bm25_retrieve(query: str, chunks: List[Dict[str, Any]], top_k: int = 5) -> List[Dict[str, Any]]:
    """Retrieve top_k chunks via BM25 sparse keyword search. Identical logic
    to app.py's bm25_retrieve.
    """
    if bm25_index is None or not chunks:
        return []
    query_tokens = _tokenize(query)
    scores = bm25_index.get_scores(query_tokens)
    ranked_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]

    retrieved = []
    for idx in ranked_indices:
        if scores[idx] <= 0:
            continue
        chunk = chunks[idx]
        retrieved.append({
            "chunk_id": chunk["chunk_id"],
            "text": chunk["text"],
            "source_file": chunk["source_file"],
            "page": chunk["page"],
            "file_type": chunk["file_type"],
            "score": float(scores[idx]),
            "retriever": "bm25",
        })
    return retrieved


## 14. Hybrid Retriever

In [13]:
def _min_max_normalize(results: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """Min-max normalize scores within a result set to [0, 1]. Falls back to
    a constant 1.0 when all scores are equal, avoiding divide-by-zero.
    """
    if not results:
        return results
    scores = [r["score"] for r in results]
    lo, hi = min(scores), max(scores)
    spread = hi - lo
    for r in results:
        r["norm_score"] = 1.0 if spread == 0 else (r["score"] - lo) / spread
    return results


def hybrid_retrieve(query: str, chunks: List[Dict[str, Any]], top_k: int = 5) -> List[Dict[str, Any]]:
    """Combine dense and BM25 retrieval into a single ranked, deduplicated
    list using min-max normalized, weighted score fusion. Identical logic
    to app.py's hybrid_retrieve.
    """
    dense_results = dense_retrieve(query, top_k=top_k * 2)
    bm25_results = bm25_retrieve(query, chunks, top_k=top_k * 2)

    dense_results = _min_max_normalize(dense_results)
    bm25_results = _min_max_normalize(bm25_results)

    combined: Dict[str, Dict[str, Any]] = {}
    for r in dense_results:
        combined[r["chunk_id"]] = dict(r, weighted_score=r["norm_score"] * config.DENSE_WEIGHT)
    for r in bm25_results:
        if r["chunk_id"] in combined:
            combined[r["chunk_id"]]["weighted_score"] += r["norm_score"] * config.BM25_WEIGHT
            combined[r["chunk_id"]]["retriever"] = "hybrid"
        else:
            combined[r["chunk_id"]] = dict(r, weighted_score=r["norm_score"] * config.BM25_WEIGHT)

    ranked = sorted(combined.values(), key=lambda r: r["weighted_score"], reverse=True)
    return ranked[:top_k]


# ---- CrossEncoder Reranking -------------------------------------------------
reranker_model = CrossEncoder(config.RERANKER_MODEL)
logger.info(f"Loaded reranker model: {config.RERANKER_MODEL}")


def rerank_chunks(query: str, chunks: List[Dict[str, Any]], top_n: int = None) -> List[Dict[str, Any]]:
    """Rerank hybrid-retrieved chunks with a CrossEncoder for much higher
    precision than embedding similarity alone. Falls back to the input
    order (already hybrid-ranked) if the reranker fails to run. Identical
    logic to app.py's rerank_chunks.
    """
    top_n = top_n or config.RERANK_TOP_N
    if not chunks:
        return chunks
    try:
        pairs = [[query, c["text"]] for c in chunks]
        scores = reranker_model.predict(pairs)
        for chunk, score in zip(chunks, scores):
            chunk["rerank_score"] = float(score)
        ranked = sorted(chunks, key=lambda c: c["rerank_score"], reverse=True)
        return ranked[:top_n]
    except Exception:
        logger.exception("Reranking failed; falling back to hybrid order")
        return chunks[:top_n]


2026-07-10 08:46:18,627 | INFO     | sentence_transformers.base.model | No device provided, using cpu
2026-07-10 08:46:19,074 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L-6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-07-10 08:46:19,542 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 404 Not Found"
2026-07-10 08:46:19,549 | INFO     | sentence_transformers.base.model | No modules.json found for cross-encoder/ms-marco-MiniLM-L-6-v2, initializing a new CrossEncoder model.
2026-07-10 08:46:20,009 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/models/cross-encoder/ms-marco-MiniLM-L-6-v2 "HTTP/1.1 307 Temporary Redirect"
2026-07-10 08:46:20,254 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/models/cross-encoder/ms-marco-MiniLM-L6-v2 "HTTP/1.1 200 OK"
2026-07-10 08:46:20,661 | INFO     | httpx | H

## 15. Adaptive RAG Router

In [14]:
ROUTER_SYSTEM_PROMPT = (
    "You are a query classification router for a document question-answering "
    "system. Classify the user's query into EXACTLY ONE of these labels:\n"
    "- Simple: a direct factual lookup answerable from one short passage.\n"
    "- Complex: requires synthesizing multiple ideas or reasoning steps.\n"
    "- Comparison: asks to compare, contrast, or weigh two or more things.\n"
    "- Summarization: asks for an overview, summary, or high-level recap.\n"
    "- Reasoning: asks 'why' or requires inference beyond stated facts.\n"
    "- Multi-document: explicitly or implicitly spans multiple documents.\n"
    "- Follow-up: only makes sense in light of prior conversation turns.\n"
    "Respond with ONLY the single label text, nothing else."
)


def _heuristic_classify_query(query: str) -> str:
    """Fast, dependency-free fallback classifier used only if the LLM
    router call fails (e.g. Groq outage), so retrieval never hard-stops.
    Identical logic to app.py's _heuristic_classify_query.
    """
    q = query.strip().lower()
    word_count = len(q.split())
    if q.split() and q.split()[0] in ("it", "that", "this", "they", "them", "those"):
        return "Follow-up"
    if any(m in q for m in ("compare", "difference", "versus", " vs ", "contrast")):
        return "Comparison"
    if any(m in q for m in ("summarize", "summary", "overview", "recap")):
        return "Summarization"
    if any(m in q for m in ("why", "explain", "reasoning", "how come")):
        return "Reasoning"
    if word_count > 20 or "across" in q or "all documents" in q:
        return "Multi-document"
    if word_count <= 3:
        return "Follow-up"
    return "Simple"


def classify_query(query: str) -> str:
    """Classify a query into one of QUERY_LABELS using the Groq LLM router,
    with a heuristic fallback if the call fails for any reason. Identical
    logic to app.py's classify_query.
    """
    try:
        response = groq_client.chat.completions.create(
            model=config.GROQ_MODEL,
            temperature=0.0,
            messages=[
                {"role": "system", "content": ROUTER_SYSTEM_PROMPT},
                {"role": "user", "content": f"Query: {query}\n\nLabel:"},
            ],
        )
        label = response.choices[0].message.content.strip()
        for known in QUERY_LABELS:
            if known.lower() == label.lower().strip(". "):
                return known
        logger.warning(f"Router returned unrecognized label '{label}'; using heuristic fallback")
        return _heuristic_classify_query(query)
    except Exception:
        logger.exception("LLM router call failed; using heuristic fallback")
        return _heuristic_classify_query(query)


def adaptive_retrieve(query: str, chunks: List[Dict[str, Any]]) -> Tuple[List[Dict[str, Any]], str]:
    """Route the query through the adaptive LLM router, retrieve via hybrid
    search, then rerank with the CrossEncoder for final precision ordering.
    Identical logic to app.py's adaptive_retrieve.
    """
    query_type = classify_query(query)
    top_k = ROUTER_TOP_K.get(query_type, 5)
    logger.info(f"Adaptive router classified query as '{query_type}' (top_k={top_k})")

    hybrid_results = hybrid_retrieve(query, chunks, top_k=max(top_k, config.RERANK_CANDIDATE_POOL))
    reranked = rerank_chunks(query, hybrid_results, top_n=min(top_k, config.RERANK_TOP_N))
    return reranked, query_type


## 16. Contextual Compression

In [15]:
def compress_context(query: str, retrieved_chunks: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """Use the LLM to extract only the sentences relevant to the query from
    each of the top-N reranked chunks, discarding irrelevant filler
    content. Only the top COMPRESSION_TOP_N chunks are compressed; any
    remaining lower-ranked chunks pass through unmodified. Identical logic
    to app.py's compress_context.
    """
    if not retrieved_chunks:
        return retrieved_chunks

    to_compress = retrieved_chunks[:config.COMPRESSION_TOP_N]
    passthrough = retrieved_chunks[config.COMPRESSION_TOP_N:]

    numbered_chunks = "\n\n".join(
        f"[CHUNK {i}]\n{c['text']}" for i, c in enumerate(to_compress)
    )

    system_prompt = (
        "You are a context compression engine. For each numbered CHUNK, extract ONLY "
        "the sentences or phrases directly relevant to the user's question. If a chunk "
        "has nothing relevant, output an empty string for it. "
        "Respond ONLY with a JSON object mapping chunk index (as string) to the extracted text, "
        "with no preamble, no markdown fences, and no extra commentary."
    )
    user_prompt = f"Question: {query}\n\n{numbered_chunks}"

    try:
        response = groq_client.chat.completions.create(
            model=config.GROQ_MODEL,
            temperature=config.GROQ_TEMPERATURE,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
        )
        raw = response.choices[0].message.content.strip()
        raw = re.sub(r"^```(json)?|```$", "", raw, flags=re.MULTILINE).strip()
        extracted_map = json.loads(raw)
    except Exception:
        logger.exception("Contextual compression failed; falling back to raw chunks")
        return retrieved_chunks

    compressed = []
    for i, chunk in enumerate(to_compress):
        extracted = extracted_map.get(str(i), "").strip()
        new_chunk = dict(chunk)
        new_chunk["text"] = extracted if extracted else chunk["text"]
        compressed.append(new_chunk)
    return compressed + passthrough


## 17. Conversation Memory

In [16]:
class ConversationMemory:
    """Holds the last config.MEMORY_WINDOW conversation turns, exactly like
    app.py's `deque(maxlen=MAX_CONVERSATION_TURNS)`.
    """

    def __init__(self, window: int = None):
        self.window = window or config.MEMORY_WINDOW
        self.turns: deque = deque(maxlen=self.window)

    def add(self, user_msg: str, assistant_msg: str) -> None:
        self.turns.append({"user": user_msg, "assistant": assistant_msg})

    def as_text(self) -> str:
        if not self.turns:
            return ""
        lines = []
        for t in self.turns:
            lines.append(f"User: {t['user']}")
            lines.append(f"Assistant: {t['assistant']}")
        return "\n".join(lines)

    def clear(self) -> None:
        self.turns.clear()

    def __bool__(self) -> bool:
        return len(self.turns) > 0


memory = ConversationMemory()


## 18. Query Rewriter

In [17]:
STANDALONE_CHECK_PROMPT = (
    "Determine whether the following follow-up question is already a fully "
    "standalone question that can be understood WITHOUT the conversation "
    "history (i.e. it contains no pronouns or references like 'it', 'that', "
    "'those', 'the previous one' that depend on prior context). "
    "Respond with ONLY 'YES' if it is standalone, or 'NO' if it depends on "
    "the conversation history."
)

REWRITE_SYSTEM_PROMPT = (
    "Given the conversation history and a follow-up question, rewrite the follow-up "
    "question to be a fully standalone question that can be understood without the "
    "conversation history. Preserve the original intent. Respond with ONLY the rewritten "
    "question and nothing else."
)


def _needs_rewrite(query: str) -> bool:
    """Ask the LLM whether the query is already standalone; skip the more
    expensive rewrite call entirely when it is. Falls back to 'needs
    rewrite' on any error (the safer default). Identical logic to app.py's
    _needs_rewrite.
    """
    try:
        response = groq_client.chat.completions.create(
            model=config.GROQ_MODEL,
            temperature=0.0,
            messages=[
                {"role": "system", "content": STANDALONE_CHECK_PROMPT},
                {"role": "user", "content": query},
            ],
        )
        verdict = response.choices[0].message.content.strip().upper()
        return not verdict.startswith("YES")
    except Exception:
        logger.exception("Standalone-check failed; defaulting to rewriting")
        return True


def rewrite_query(query: str, history_text: str) -> str:
    """Rewrite the user's query into a standalone question using
    conversation history, but only when the query is not already
    standalone. Identical logic to app.py's rewrite_query.
    """
    if not memory:
        return query

    if not _needs_rewrite(query):
        logger.info(f"Query already standalone; skipping rewrite: '{query}'")
        return query

    user_prompt = f"Conversation history:\n{history_text}\n\nFollow-up question: {query}\n\nStandalone question:"

    try:
        response = groq_client.chat.completions.create(
            model=config.GROQ_MODEL,
            temperature=config.GROQ_TEMPERATURE,
            messages=[
                {"role": "system", "content": REWRITE_SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt},
            ],
        )
        rewritten = response.choices[0].message.content.strip()
        logger.info(f"Rewrote query: '{query}' -> '{rewritten}'")
        return rewritten if rewritten else query
    except Exception:
        logger.exception("Query rewriting failed; using original query")
        return query


## 19. Prompt Template

In [18]:
ANSWER_SYSTEM_PROMPT = (
    "You are DocSensei, an enterprise document assistant. You must answer ONLY using the "
    "provided context extracted from the user's uploaded documents. Never use outside or "
    "general knowledge. If the answer cannot be found in the provided context, respond "
    "exactly with: \"I could not find this information in the uploaded documents.\" "
    "Write the answer naturally, in plain prose. Do NOT include citation markers, source "
    "names, or bracketed references inside the answer text itself; citations are handled "
    "separately."
)


## 20. Groq LLM

In [19]:
groq_client = Groq(api_key=GROQ_API_KEY)
logger.info(f"Groq client initialized: {config.GROQ_MODEL}")


2026-07-10 08:47:06,662 | INFO     | DocSensei | Groq client initialized: llama-3.3-70b-versatile


## 21. Complete Adaptive RAG Pipeline

In [20]:
@dataclass
class RAGSession:
    chunks: List[Dict[str, Any]] = field(default_factory=list)
    raw_documents: Dict[str, str] = field(default_factory=dict)
    summaries: Dict[str, Any] = field(default_factory=dict)
    suggested_questions: List[Dict[str, str]] = field(default_factory=list)

    def ingest(self, filepaths: List[str]) -> None:
        """Ingest a new batch of documents.

        Every upload starts a completely new RAG session: previous
        Pinecone vectors, BM25 index, chunks, and conversation memory are
        all cleared first, so retrieval can never surface a deleted
        document. Identical flow to app.py's /upload route.
        """
        logger.info("Starting new ingestion - clearing previous session state.")
        delete_all_vectors()
        memory.clear()
        self.chunks = []
        self.raw_documents = {}
        self.summaries = {}
        self.suggested_questions = []

        with timeit("Loading"):
            loaded = load_documents(filepaths)

        if not loaded:
            logger.warning("No documents were successfully loaded; ingestion aborted.")
            return

        self.raw_documents = build_raw_documents(loaded)

        with timeit("Chunking"):
            self.chunks = chunk_documents(loaded)

        if not self.chunks:
            logger.warning("No extractable text found in the uploaded files.")
            return

        with timeit("Embedding + Pinecone indexing"):
            upsert_chunks_to_pinecone(self.chunks)

        build_bm25_index(self.chunks)

        with timeit("Summarization"):
            summaries: Dict[str, Any] = {}
            for filename, text in self.raw_documents.items():
                try:
                    summaries[filename] = summarize_document(filename, text)
                except Exception:
                    logger.exception(f"Summary generation failed for '{filename}'")
                    summaries[filename] = _fallback_summary(text)
            self.summaries = summaries

        with timeit("Suggested questions"):
            combined_sample = "\n\n".join(self.raw_documents.values())
            try:
                self.suggested_questions = generate_suggested_questions(combined_sample)
            except Exception:
                logger.exception("Suggested question generation failed for upload batch")
                self.suggested_questions = []

        num_files = len(self.raw_documents)
        logger.info(f"Ingestion complete: {len(self.chunks)} chunks from {num_files} file(s).")

    def answer(self, question: str) -> Dict[str, Any]:
        """Identical flow to app.py's /chat route:
        rewrite -> adaptive retrieve -> compress -> generate -> citations.
        """
        if not self.chunks:
            return {
                "answer": "Please upload one or more documents before asking questions.",
                "citations": [],
            }

        try:
            # 1. Rewrite query using conversation history (skipped if already standalone)
            history_text = memory.as_text()
            standalone_query = rewrite_query(question, history_text)

            # 2. Adaptive retrieval: LLM router -> hybrid retrieve -> CrossEncoder rerank
            retrieved_chunks, query_type = adaptive_retrieve(standalone_query, self.chunks)

            # 3. Contextual compression (top-N only)
            compressed_chunks = compress_context(standalone_query, retrieved_chunks)

            # 4. Generate answer
            answer_text = generate_answer(standalone_query, compressed_chunks)

            # 5. Build citations (returned separately, never injected into the answer)
            citations = build_citations(compressed_chunks)

            # 6. Update conversation memory
            memory.add(question, answer_text)

            return {
                "answer": answer_text,
                "citations": citations,
                "query_type": query_type,
                "rewritten_query": standalone_query,
            }

        except Exception:
            logger.exception("Error while answering question")
            return {
                "answer": "An error occurred while generating the answer. Please try again.",
                "citations": [],
            }


def generate_answer(query: str, context_chunks: List[Dict[str, Any]]) -> str:
    """Generate a natural-language answer grounded strictly in the
    retrieved context. Temperature is fixed at config.GROQ_TEMPERATURE
    (0.1) to minimize hallucination. Identical logic to app.py's
    generate_answer.
    """
    if not context_chunks:
        return "I could not find this information in the uploaded documents."

    context_text = "\n\n---\n\n".join(c["text"] for c in context_chunks if c["text"].strip())
    if not context_text.strip():
        return "I could not find this information in the uploaded documents."

    user_prompt = f"Context from uploaded documents:\n{context_text}\n\nQuestion: {query}\n\nAnswer:"

    try:
        response = groq_client.chat.completions.create(
            model=config.GROQ_MODEL,
            temperature=config.GROQ_TEMPERATURE,
            messages=[
                {"role": "system", "content": ANSWER_SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt},
            ],
        )
        return response.choices[0].message.content.strip()
    except Exception:
        logger.exception("Answer generation failed")
        return "An error occurred while generating the answer. Please try again."


def build_citations(context_chunks: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """Build a separate citations list referencing source file, page/slide,
    and chunk id. Citations are never injected into the answer text
    itself. Identical logic to app.py's build_citations.
    """
    citations = []
    seen = set()
    for chunk in context_chunks:
        key = (chunk.get("source_file"), chunk.get("page"), chunk.get("chunk_id"))
        if key in seen:
            continue
        seen.add(key)
        snippet = chunk.get("text", "")[:220]
        citations.append({
            "source_file": chunk.get("source_file"),
            "page": chunk.get("page"),
            "chunk_id": chunk.get("chunk_id"),
            "file_type": chunk.get("file_type"),
            "snippet": snippet,
        })
    return citations


session = RAGSession()


## 22. Document Summarizer

In [21]:
SUMMARY_SYSTEM_PROMPT = (
    "You are a precise document summarization assistant. Summarize the given document "
    "into a structured JSON object with EXACTLY these keys:\n"
    '  "executive_summary": 2-4 sentence high-level overview.\n'
    '  "key_topics": array of 3-7 short topic strings.\n'
    '  "important_facts": array of 3-6 concise factual statement strings.\n'
    '  "key_numbers": array of notable figures/statistics/dates as strings '
    "(empty array if the document has none).\n"
    '  "conclusion": 1-3 sentence closing takeaway.\n'
    "Respond with ONLY the JSON object, no preamble, no markdown fences."
)


def _fallback_summary(text: str) -> Dict[str, Any]:
    """A minimal structured summary used only if the LLM call fails, so
    downstream code never has to special-case a missing summary.
    """
    return {
        "executive_summary": "Summary unavailable due to an internal error.",
        "key_topics": [],
        "important_facts": [],
        "key_numbers": [],
        "conclusion": "",
    }


def summarize_document(filename: str, full_text: str) -> Dict[str, Any]:
    """Generate a structured summary for a single uploaded document,
    covering an executive summary, key topics, important facts, key
    numbers, and a conclusion. Identical logic to app.py's
    summarize_document.
    """
    truncated_text = full_text[:12000]  # guard against oversized prompts
    user_prompt = f"Document: {filename}\n\nContent:\n{truncated_text}\n\nJSON summary:"

    try:
        response = groq_client.chat.completions.create(
            model=config.GROQ_MODEL,
            temperature=config.GROQ_TEMPERATURE,
            messages=[
                {"role": "system", "content": SUMMARY_SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt},
            ],
        )
        raw = response.choices[0].message.content.strip()
        raw = re.sub(r"^```(json)?|```$", "", raw, flags=re.MULTILINE).strip()
        parsed = json.loads(raw)
        return {
            "executive_summary": parsed.get("executive_summary", ""),
            "key_topics": parsed.get("key_topics", []) or [],
            "important_facts": parsed.get("important_facts", []) or [],
            "key_numbers": parsed.get("key_numbers", []) or [],
            "conclusion": parsed.get("conclusion", ""),
        }
    except Exception:
        logger.exception(f"Summarization failed for file: {filename}")
        return _fallback_summary(full_text)


## 23. Suggested Questions

In [22]:
SUGGESTED_QUESTIONS_SYSTEM_PROMPT = (
    "Based on the provided document content, generate exactly 5 insightful questions "
    "that a user could ask and that would be answerable from this content, one for "
    "EACH of the following difficulty/category levels, in this order: "
    "Basic, Intermediate, Advanced, Comparison, Analytical. "
    'Respond ONLY with a JSON array of 5 objects, each shaped as '
    '{"category": "...", "question": "..."}, no preamble, no markdown fences.'
)


def generate_suggested_questions(all_text_sample: str) -> List[Dict[str, str]]:
    """Generate exactly 5 suggested questions spanning Basic, Intermediate,
    Advanced, Comparison, and Analytical categories. Identical logic to
    app.py's generate_suggested_questions.
    """
    truncated_text = all_text_sample[:12000]
    user_prompt = f"Content:\n{truncated_text}"

    try:
        response = groq_client.chat.completions.create(
            model=config.GROQ_MODEL,
            temperature=config.GROQ_TEMPERATURE,
            messages=[
                {"role": "system", "content": SUGGESTED_QUESTIONS_SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt},
            ],
        )
        raw = response.choices[0].message.content.strip()
        raw = re.sub(r"^```(json)?|```$", "", raw, flags=re.MULTILINE).strip()
        questions = json.loads(raw)
        if isinstance(questions, list):
            cleaned = []
            for q in questions[:5]:
                if isinstance(q, dict) and q.get("question"):
                    cleaned.append({"category": q.get("category", ""), "question": q["question"]})
                elif isinstance(q, str):
                    cleaned.append({"category": "", "question": q})
            return cleaned
        return []
    except Exception:
        logger.exception("Suggested question generation failed")
        return []


In [23]:
def reset_session() -> None:
    """Completely reset the application state: wipe all Pinecone vectors,
    clear BM25, chunks, conversation memory, and any cached summaries or
    suggested questions. Call this any time you want a clean slate without
    immediately re-ingesting new documents. Identical intent to app.py's
    /reset route.
    """
    delete_all_vectors()
    memory.clear()
    session.chunks = []
    session.raw_documents = {}
    session.summaries = {}
    session.suggested_questions = []
    build_bm25_index([])
    logger.info("Reset: application state fully cleared.")


In [28]:
documents = [
    r"pnu.pptx"
]

session.ingest(documents)

print("Documents indexed successfully!")
print(f"Total chunks: {len(session.chunks)}")

if session.summaries:
    print("\nDocument Summaries:")
    for source, summary in session.summaries.items():
        print(f"\n- {source}:")
        print(f"  Executive summary: {summary.get('executive_summary', '')}")
        if summary.get("key_topics"):
            print(f"  Key topics: {', '.join(summary['key_topics'])}")
        if summary.get("important_facts"):
            print("  Important facts:")
            for fact in summary["important_facts"]:
                print(f"    - {fact}")
        if summary.get("key_numbers"):
            print(f"  Key numbers: {', '.join(summary['key_numbers'])}")
        if summary.get("conclusion"):
            print(f"  Conclusion: {summary['conclusion']}")

if session.suggested_questions:
    print("\nSuggested Questions:")
    for q in session.suggested_questions:
        label = f"{q['category']}: " if q.get("category") else ""
        print(f"  - {label}{q['question']}")


2026-07-10 08:48:10,513 | INFO     | DocSensei | Starting new ingestion - clearing previous session state.
2026-07-10 08:48:11,082 | WARNING  | DocSensei | Pinecone delete_all warning (likely an empty index): (404)
Reason: Not Found
HTTP response headers: HTTPHeaderDict({'Date': 'Fri, 10 Jul 2026 03:18:10 GMT', 'Content-Type': 'application/json', 'Content-Length': '55', 'Connection': 'keep-alive', 'x-pinecone-request-latency-ms': '51', 'x-envoy-upstream-service-time': '52', 'x-pinecone-response-duration-ms': '53', 'server': 'envoy'})
HTTP response body: {"code":5,"message":"Namespace not found","details":[]}

2026-07-10 08:48:11,082 | INFO     | DocSensei | Loading pnu.pptx (.pptx)
2026-07-10 08:48:11,218 | INFO     | DocSensei | Loaded PPTX 'pnu.pptx' with 19 slides of text
2026-07-10 08:48:11,220 | INFO     | DocSensei | Loaded 1 file(s) out of 1 requested.
2026-07-10 08:48:11,222 | INFO     | DocSensei | Loading took 0.14s
2026-07-10 08:48:11,226 | INFO     | DocSensei | Built 21 ch

Documents indexed successfully!
Total chunks: 21

Document Summaries:

- pnu.pptx:
  Executive summary: Summary unavailable due to an internal error.

Suggested Questions:
  - Basic: What percentage of deaths in children under five is caused by pneumonia?
  - Intermediate: What techniques were used for noise reduction and contrast enhancement in the image preprocessing pipeline?
  - Advanced: How does the use of transfer learning in the VGG16 model contribute to its high accuracy in pneumonia classification?
  - Comparison: How does the performance of the custom CNN model compare to that of the fine-tuned VGG16 and DenseNet169 models in terms of recall and precision?
  - Analytical: What are the implications of using a unified framework for image preparation and model classification in terms of fairness and consistency in evaluating different deep learning architectures?


In [29]:
while True:
    question = input("\nAsk your question (type 'exit' to quit): ")

    if question.lower() == "exit":
        break

    result = session.answer(question)

    print("\n" + "=" * 80)
    print("ANSWER:\n")
    print(result["answer"])

    if result.get("citations"):
        print("\nSources:")
        for c in result["citations"]:
            page_part = f", page {c['page']}" if c.get("page") is not None else ""
            print(f"  [{c['source_file']}{page_part}]")



Ask your question (type 'exit' to quit):  what model is used


2026-07-10 08:48:32,755 | INFO     | httpx | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-07-10 08:48:32,757 | INFO     | DocSensei | Adaptive router classified query as 'Simple' (top_k=3)
Batches: 100%|██████████| 1/1 [00:00<00:00,  1.80it/s]
2026-07-10 08:48:34,703 | INFO     | httpx | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-07-10 08:48:35,165 | INFO     | httpx | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"



ANSWER:

The models used include a Custom CNN with a 5-block architecture, VGG16, and DenseNet169. The Custom CNN has 32 to 512 filters and is designed to learn local patterns from radiographs. VGG16 is a deep model that uses pre-trained ImageNet weights, with 8 convolution layers unfrozen for task-specific feature learning. DenseNet169 is utilized for its dense connectivity, with 50 layers unfrozen for adaptation.

Sources:
  [pnu.pptx, page 9.0]
  [pnu.pptx, page 16.0]
  [pnu.pptx, page 3.0]



Ask your question (type 'exit' to quit):  out of all models what is best model


2026-07-10 08:48:53,279 | INFO     | httpx | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-07-10 08:48:53,524 | INFO     | httpx | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-07-10 08:48:53,525 | INFO     | DocSensei | Rewrote query: 'out of all models what is best model' -> 'What is the most effective deep learning model among Custom CNN, VGG16, and DenseNet169 for learning local patterns and task-specific feature learning from radiographs?'
2026-07-10 08:48:53,687 | INFO     | httpx | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-07-10 08:48:53,688 | INFO     | DocSensei | Adaptive router classified query as 'Comparison' (top_k=8)
Batches: 100%|██████████| 1/1 [00:00<00:00,  1.27it/s]
2026-07-10 08:48:55,954 | INFO     | httpx | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-07-10 08:48:56,659 | INFO     | htt


ANSWER:

The VGG16 Fine-Tuned model is the most effective, achieving the highest accuracy of 0.9119 and F1 Score of 0.9332, as well as an AUC of 0.9757, indicating excellent ability to separate Normal and Pneumonia classes. This suggests that transfer learning using VGG16, combined with localised contrast enhancement, provides the most reliable pneumonia screening.

Sources:
  [pnu.pptx, page 9.0]
  [pnu.pptx, page 11.0]
  [pnu.pptx, page 15.0]
  [pnu.pptx, page 12.0]
  [pnu.pptx, page 3.0]
  [pnu.pptx, page 4.0]
  [pnu.pptx, page 18.0]
  [pnu.pptx, page 18.0]



Ask your question (type 'exit' to quit):  exit


In [26]:
# To fully wipe the current session (Pinecone vectors, BM25, chunks,
# conversation memory, summaries, and suggested questions) without
# immediately ingesting a new batch of documents, run:
# reset_session()
